In [ ]:
import FinanceDataReader as fdr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

class CustomOscillatorStrategy:
    def __init__(self, ticker, name):
        self.ticker = ticker
        self.name = name
        self.df = None

    def fetch_data(self, start_date, end_date):
        print(f"[{self.name}] 데이터 수집 중...")
        self.df = fdr.DataReader(self.ticker, start_date, end_date)
        return self.df

    def calculate_indicators(self):
        """지표 계산: RSI(14), Stochastic Slow(14, 3, 3)"""
        # 1. RSI (14일)
        delta = self.df['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        self.df['RSI'] = 100 - (100 / (1 + (gain / loss)))

        # 2. Stochastic Slow (14, 3, 3)
        # Fast K 계산
        low_min = self.df['Low'].rolling(window=14).min()
        high_max = self.df['High'].rolling(window=14).max()
        self.df['fast_k'] = (self.df['Close'] - low_min) / (high_max - low_min) * 100
        
        # Slow K (%K): Fast K의 3일 이동평균
        self.df['slow_k'] = self.df['fast_k'].rolling(window=3).mean()
        # Slow D (%D): Slow K의 3일 이동평균
        self.df['slow_d'] = self.df['slow_k'].rolling(window=3).mean()

    def run_backtest(self):
        """사용자 지정 전략 실행"""
        self.df['Signal'] = 0
        position = 0
        signals = []

        # 골든크로스/데드크로스 판단을 위해 이전 값 필요
        for i in range(len(self.df)):
            if i < 1: 
                signals.append(0)
                continue

            rsi = self.df['RSI'].iloc[i]
            k_curr = self.df['slow_k'].iloc[i]
            d_curr = self.df['slow_d'].iloc[i]
            k_prev = self.df['slow_k'].iloc[i-1]
            d_prev = self.df['slow_d'].iloc[i-1]

            # 1. 매수 로직: RSI 50~70 사이 + 스토캐스틱 20 이하 골든크로스
            if position == 0:
                if (50 <= rsi < 70) and (k_prev <= d_prev and k_curr > d_curr) and (k_curr <= 20):
                    position = 1
            
            # 2. 매도 로직: RSI 50 이하 + 스토캐스틱 80 이상 데드크로스
            elif position == 1:
                if (rsi <= 50) and (k_prev >= d_prev and k_curr < d_curr) and (k_curr >= 80):
                    position = 0
            
            signals.append(position)
        
        self.df['Position'] = signals
        
        # 수익률 및 MDD 계산
        self.df['Daily_Return'] = self.df['Close'].pct_change()
        self.df['Strategy_Return'] = (1 + (self.df['Daily_Return'] * self.df['Position'].shift(1).fillna(0))).cumprod()
        self.df['Market_Return'] = (1 + self.df['Daily_Return'].fillna(0)).cumprod()
        
        self.df['Peak'] = self.df['Strategy_Return'].cummax()
        self.df['DD'] = (self.df['Strategy_Return'] / self.df['Peak']) - 1
        
        return self.get_summary()

    def get_summary(self):
        s_ret = (self.df['Strategy_Return'].iloc[-1] - 1) * 100
        m_ret = (self.df['Market_Return'].iloc[-1] - 1) * 100
        mdd = self.df['DD'].min() * 100
        return {"전략수익률": s_ret, "시장수익률": m_ret, "MDD": mdd}

    def plot_results(self):
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 15), sharex=True)

        # 수익률 비교
        ax1.plot(self.df.index, self.df['Strategy_Return'], label='전략', color='royalblue', lw=2)
        ax1.plot(self.df.index, self.df['Market_Return'], label='시장(존버)', color='silver', alpha=0.6)
        ax1.set_title(f"{self.name} 전략 결과", fontsize=15)
        ax1.legend()

        # RSI 지표
        ax2.plot(self.df.index, self.df['RSI'], label='RSI', color='forestgreen')
        ax2.axhline(70, color='red', ls='--', alpha=0.5)
        ax2.axhline(50, color='orange', ls='--', alpha=0.5)
        ax2.axhline(30, color='blue', ls='--', alpha=0.5)
        ax2.set_title("RSI (50-70 구간 확인)")
        ax2.legend()

        # Stochastic 지표
        ax3.plot(self.df.index, self.df['slow_k'], label='%K', color='black')
        ax3.plot(self.df.index, self.df['slow_d'], label='%D', color='red', ls='--')
        ax3.axhline(80, color='gray', ls=':', alpha=0.7)
        ax3.axhline(20, color='gray', ls=':', alpha=0.7)
        ax3.set_title("Stochastic Slow (14, 3, 3)")
        ax3.legend()

        plt.tight_layout()
        plt.show()

# --- 실행 ---
analyzer = CustomOscillatorStrategy('005420', '코스모화학')
analyzer.fetch_data('2010-01-01', '2026-1-31')
analyzer.calculate_indicators()
res = analyzer.run_backtest()

print(f"📊 분석 리포트")
print(f"▶ 최종 전략 수익률: {res['전략수익률']:.2f}%")
print(f"▶ 시장 단순 보유 수익률: {res['시장수익률']:.2f}%")
print(f"▶ 전략 최대 낙폭(MDD): {res['MDD']:.2f}%")

analyzer.plot_results()

[코스모화학] 데이터 수집 중...
📊 분석 리포트
▶ 최종 전략 수익률: 65.28%
▶ 시장 단순 보유 수익률: 191.32%
▶ 전략 최대 낙폭(MDD): -85.45%
